In [36]:
from SPARQLWrapper import SPARQLWrapper, JSON, CONSTRUCT, TURTLE
from rdflib.plugins.sparql import prepareQuery
from rdflib.namespace import RDFS, URIRef

import configparser
import rdflib
import pandas as pd

# since rdflib is not working properly with SPARQL construct, we use an alternative way using GraphDB for queries
# ... to use, please do the following steps: 
# 
# 1) change the flag below to True
# 2) prepare a SPARQL endpoint on the server and replace the sparql_endpoint name below
# 3) load both OCED ontology and resulted RDF file (in this case oced_ontology.ttl and 2013_small.ttl) to the prepared SPARQL endpoint


config = configparser.ConfigParser()
config.read("config.ini")

use_endpoint = config.get('config', 'use_endpoint')
sparql_endpoint = config.get('config', 'sparql_endpoint')

OCEDO_FIlENAME = config.get('config', 'oced_ontology')
ext_FILENAME = config.get('config', 'ext_ontology')
INPUT_FILENAME =  config.get('config', 'trace_ttl')

OBJECT_OBJECT_MAP_FILENAME =  config.get('config', 'map_object-object')
EVENT_OBJECT_MAP_FILENAME =  config.get('config', 'map_event-object')

EVENT_SUPERCLASS =  config.get('config', 'event_superclass')
OBJECT_SUPERCLASS =  config.get('config', 'object_superclass')

OUTPUT_FILENAME = config.get('config', 'enriched_trace_ttl')

In [37]:
# prepare rdflib graph
ontology_graph = rdflib.Graph()

oced = rdflib.Namespace('https://w3id.org/ocedo/core#')
ext = rdflib.Namespace('https://w3id.org/ocedo/ext#')
aux = rdflib.Namespace('https://w3id.org/ocedo/aux#')
res = rdflib.Namespace('https://w3id.org/ocedo/resource/')
ontology_graph.bind('oced', oced)
ontology_graph.bind('ext', ext)
ontology_graph.bind('res', res)
ontology_graph.bind('aux', aux)

# load OCED ontology
ontology_graph.parse(OCEDO_FIlENAME, format="turtle")

# load input graph
input_graph = rdflib.Graph()
input_graph.bind('oced', oced)
input_graph.bind('ext', ext)
input_graph.bind('res', res)
input_graph.bind('aux', aux)
input_graph.parse(INPUT_FILENAME, format="turtle")
input_graph += ontology_graph


In [38]:
# prepare construct query for event_object enhancement
# -- parameters needed: $object_type, $ext_class, $ext_relation
event_object_cq = """
prefix oced: <https://w3id.org/ocedo/core#>
prefix ext: <https://w3id.org/ocedo/ext#>
prefix aux: <https://w3id.org/ocedo/aux#>
prefix res: <https://w3id.org/ocedo/resource/>
prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#>
prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>

CONSTRUCT {
    ?object a $ext_class .
    ?event $ext_relation ?object .
} 
WHERE {
    ?eo a aux:Observe ;
    	aux:eo_event ?event ;
    	aux:eo_object ?object ;
    .
    ?object aux:instance_of/rdfs:label "$object_type" .
}
"""

# prepare construct query for object_object enhancement
# -- parameters needed: $object1_class, $object2_class, $object1_type, $object2_type, $ext_relation
object_object_cq = """
prefix oced: <https://w3id.org/ocedo/core#>
prefix ext: <https://w3id.org/ocedo/ext#>
prefix aux: <https://w3id.org/ocedo/aux#>
prefix res: <https://w3id.org/ocedo/resource/>
prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#>

CONSTRUCT {
    ?object1 a $object1_class .
    ?object2 a $object2_class .
    ?object1 $ext_relation ?object2 .
} 
WHERE {
    ?eo1 a aux:Observe ;
    	aux:eo_event ?event ;
    	aux:eo_object ?object1 ;
    .
    ?eo2 a aux:Observe ;
    	aux:eo_event ?event ;
    	aux:eo_object ?object2 ;
    .
    ?object1 aux:instance_of/rdfs:label "$object1_type" .
    ?object2 aux:instance_of/rdfs:label "$object2_type" .
}
"""

In [39]:
# prepare construct query for event_object enhancement
# -- parameters needed: $object_type, $ext_class, $ext_relation
event_object_ext_cq = """
prefix oced: <https://w3id.org/ocedo/core#>
prefix ext: <https://w3id.org/ocedo/ext#>
prefix aux: <https://w3id.org/ocedo/aux#>
prefix res: <https://w3id.org/ocedo/resource/>
prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#>
prefix owl: <http://www.w3.org/2002/07/owl#>
prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>

CONSTRUCT {
    $ext_class a owl:Class ;
        rdfs:subClassOf <$object_superclass> .
    $ext_relation a owl:ObjectProperty ;
        rdfs:domain oced:Event ;
        rdfs:range $ext_class ;
        rdfs:label "$ext_relation".
} WHERE {}
"""

# prepare construct query for object_object enhancement
# -- parameters needed: $object1_class, $object2_class, $object1_type, $object2_type, $ext_relation
object_object_ext_cq = """
prefix oced: <https://w3id.org/ocedo/core#>
prefix ext: <https://w3id.org/ocedo/ext#>
prefix aux: <https://w3id.org/ocedo/aux#>
prefix res: <https://w3id.org/ocedo/resource/>
prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#>
prefix owl: <http://www.w3.org/2002/07/owl#>

CONSTRUCT {
    $object1_class a owl:Class ;
        rdfs:subClassOf <$object_superclass> .
    $object2_class a owl:Class ;
        rdfs:subClassOf <$object_superclass> .
    $ext_relation a owl:ObjectProperty ;
        rdfs:domain $object1_class ;
        rdfs:range $object2_class ;
        rdfs:label "$ext_relation".
} WHERE {}
"""

In [40]:

def build_query_superclass(query):
    cq = query
    cq = cq.replace("$object_superclass", ext+OBJECT_SUPERCLASS)
    cq = cq.replace("$event_superclass", ext+EVENT_SUPERCLASS)
    return cq

def build_event_object_cq(query, object_type, ext_class, ext_relation):
    cq = query
    cq = cq.replace("$object_type", object_type)
    cq = cq.replace("$ext_class", ext_class)
    cq = cq.replace("$ext_relation", ext_relation)
    cq = cq.replace("$object_superclass", ext+OBJECT_SUPERCLASS)
    return cq

def build_object_object_cq(query, object1_class, object2_class, object1_type, object2_type, ext_relation):
    cq = query
    cq = cq.replace("$object1_class", object1_class)
    cq = cq.replace("$object2_class", object2_class)
    cq = cq.replace("$object1_type", object1_type)
    cq = cq.replace("$object2_type", object2_type)
    cq = cq.replace("$ext_relation", ext_relation)
    cq = cq.replace("$object_superclass", ext+OBJECT_SUPERCLASS)
    return cq

def run_construct_query_endpoint(cq):
    sparql = SPARQLWrapper(sparql_endpoint)
    sparql.setQuery(cq)
    sparql.setReturnFormat(TURTLE)
    sparql.setMethod(CONSTRUCT)
    results = sparql.queryAndConvert()

    result_graph = rdflib.Graph()
    result_graph.parse(data=results, format='ttl')
    return result_graph

def run_construct_query_rdflib(cq):
    construct_query = prepareQuery(cq)
    result_graph = input_graph.query(construct_query).graph

    return result_graph


In [41]:
temp_data_result = rdflib.Graph()
temp_ext_result = rdflib.Graph()

df = pd.read_csv(EVENT_OBJECT_MAP_FILENAME)
for index, row in df.iterrows():
    o_type = row["object_type"]
    o_class = row["ext_class"]
    o_relation = row["ext_relation"]
    cq_string = build_event_object_cq(event_object_cq, o_type, o_class, o_relation)
    ocedo_cq_string = build_event_object_cq(event_object_ext_cq, o_type, o_class, o_relation)
    print(ocedo_cq_string)
    if use_endpoint:
        temp_data_result += run_construct_query_endpoint(cq_string)
        temp_ext_result += run_construct_query_endpoint(ocedo_cq_string)
    else: 
        temp_data_result += run_construct_query_rdflib(cq_string)
        temp_ext_result += run_construct_query_rdflib(ocedo_cq_string)




prefix oced: <https://w3id.org/ocedo/core#>
prefix ext: <https://w3id.org/ocedo/ext#>
prefix aux: <https://w3id.org/ocedo/aux#>
prefix res: <https://w3id.org/ocedo/resource/>
prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#>
prefix owl: <http://www.w3.org/2002/07/owl#>
prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>

CONSTRUCT {
    ext:Product a owl:Class ;
        rdfs:subClassOf <https://w3id.org/ocedo/ext#ObjectBPIC2013> .
    ext:is_about_product a owl:ObjectProperty ;
        rdfs:domain oced:Event ;
        rdfs:range ext:Product ;
        rdfs:label "ext:is_about_product".
} WHERE {}


prefix oced: <https://w3id.org/ocedo/core#>
prefix ext: <https://w3id.org/ocedo/ext#>
prefix aux: <https://w3id.org/ocedo/aux#>
prefix res: <https://w3id.org/ocedo/resource/>
prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#>
prefix owl: <http://www.w3.org/2002/07/owl#>
prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>

CONSTRUCT {
    ext:ProductFunction a owl:Class ;
   

In [42]:
df = pd.read_csv(OBJECT_OBJECT_MAP_FILENAME)
for index, row in df.iterrows():
    object1_class = row["object1_class"]
    object2_class = row["object2_class"]
    object1_type = row["object1_type"]
    object2_type = row["object2_type"]
    ext_relation = row["ext_relation"]
    cq_string = build_object_object_cq(object_object_cq, object1_class, object2_class, object1_type, object2_type, ext_relation)
    ocedo_cq_string = build_object_object_cq(object_object_ext_cq, object1_class, object2_class, object1_type, object2_type, ext_relation)
    print(cq_string)
    if use_endpoint:
        temp_data_result += run_construct_query_endpoint(cq_string)
        temp_ext_result += run_construct_query_endpoint(ocedo_cq_string)
    else:
        temp_data_result += run_construct_query_rdflib(cq_string) # ==> not sure why, but it's really slow
        temp_ext_result += run_construct_query_rdflib(ocedo_cq_string)


prefix oced: <https://w3id.org/ocedo/core#>
prefix ext: <https://w3id.org/ocedo/ext#>
prefix aux: <https://w3id.org/ocedo/aux#>
prefix res: <https://w3id.org/ocedo/resource/>
prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#>

CONSTRUCT {
    ?object1 a ext:TeamMember .
    ?object2 a ext:Location .
    ?object1 ext:located_in ?object2 .
} 
WHERE {
    ?eo1 a aux:Observe ;
    	aux:eo_event ?event ;
    	aux:eo_object ?object1 ;
    .
    ?eo2 a aux:Observe ;
    	aux:eo_event ?event ;
    	aux:eo_object ?object2 ;
    .
    ?object1 aux:instance_of/rdfs:label "org:resource" .
    ?object2 aux:instance_of/rdfs:label "resource country" .
}


prefix oced: <https://w3id.org/ocedo/core#>
prefix ext: <https://w3id.org/ocedo/ext#>
prefix aux: <https://w3id.org/ocedo/aux#>
prefix res: <https://w3id.org/ocedo/resource/>
prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#>

CONSTRUCT {
    ?object1 a ext:TeamMember .
    ?object2 a ext:SupportTeam .
    ?object1 ext:part_of ?object2 .
} 
WHE

In [43]:
# prepare construct query for event type settings
event_type_cq = """
    prefix oced: <https://w3id.org/ocedo/core#>
    prefix ext: <https://w3id.org/ocedo/ext#>
    prefix aux: <https://w3id.org/ocedo/aux#>
    prefix res: <https://w3id.org/ocedo/resource/>
    prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    prefix owl: <http://www.w3.org/2002/07/owl#>
    prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>

    CONSTRUCT {
        ?event a ?event_type .

    } WHERE {
        ?event a oced:Event ;
            aux:instance_of/rdfs:label ?event_type_label .
        BIND( IRI(CONCAT("https://w3id.org/ocedo/ext#",REPLACE(?event_type_label, " ", "_"))) as ?event_type)
    }       
"""
event_type_class_cq = """
    prefix oced: <https://w3id.org/ocedo/core#>
    prefix ext: <https://w3id.org/ocedo/ext#>
    prefix aux: <https://w3id.org/ocedo/aux#>
    prefix res: <https://w3id.org/ocedo/resource/>
    prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    prefix owl: <http://www.w3.org/2002/07/owl#>
    prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>

    CONSTRUCT {
        ?event_type a owl:Class ; 
            rdfs:subClassOf <$event_superclass> .

    } WHERE {
        ?event a oced:Event ;
            aux:instance_of/rdfs:label ?event_type_label .
        BIND( IRI(CONCAT("https://w3id.org/ocedo/ext#",REPLACE(?event_type_label, " ", "_"))) as ?event_type)
    }       
"""

# prepare construct query for event attribute enhancement
event_attr_cq = """
    prefix oced: <https://w3id.org/ocedo/core#>
    prefix ext: <https://w3id.org/ocedo/ext#>
    prefix aux: <https://w3id.org/ocedo/aux#>
    prefix res: <https://w3id.org/ocedo/resource/>
    prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    prefix owl: <http://www.w3.org/2002/07/owl#>
    prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>

    CONSTRUCT {
        ?event ?attr_uri ?attr_value .

    } WHERE {
        ?event a oced:Event ;
            aux:hasEventAttribute/aux:event_attribute_value ?attr_value .
        ?event_attr aux:event_attribute ?attr_name ;
            aux:event_attribute_value ?attr_value .
        BIND( IRI(CONCAT("https://w3id.org/ocedo/ext#",REPLACE(REPLACE(?attr_name, " ", "_"),":", "_"))) as ?attr_uri)
    }       
"""
event_attr_property_cq = """
    prefix oced: <https://w3id.org/ocedo/core#>
    prefix ext: <https://w3id.org/ocedo/ext#>
    prefix aux: <https://w3id.org/ocedo/aux#>
    prefix res: <https://w3id.org/ocedo/resource/>
    prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    prefix owl: <http://www.w3.org/2002/07/owl#>
    prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>

    CONSTRUCT {
        ?attr_uri a owl:datatypeProperty ;
            rdfs:subPropertyOf oced:event_attribute .

    } WHERE {
        ?event a oced:Event ;
            aux:hasEventAttribute/ aux:event_attribute ?attr_name .
        BIND( IRI(CONCAT("https://w3id.org/ocedo/ext#",REPLACE(REPLACE(?attr_name, " ", "_"),":", "_"))) as ?attr_uri)
    }       
"""

# prepare construct query for object attribute enhancement
object_attr_cq = """
    prefix oced: <https://w3id.org/ocedo/core#>
    prefix ext: <https://w3id.org/ocedo/ext#>
    prefix aux: <https://w3id.org/ocedo/aux#>
    prefix res: <https://w3id.org/ocedo/resource/>
    prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    prefix owl: <http://www.w3.org/2002/07/owl#>
    prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>

    CONSTRUCT {
        ?object ?attr_uri ?attr_value .

    } WHERE {
        ?object a oced:Object ;
            aux:hasObjectAttribute ?object_attr .
        ?object_attr aux:object_attribute ?attr_name ;
            aux:object_attribute_value ?attr_value .
        BIND( IRI(CONCAT("https://w3id.org/ocedo/ext#",REPLACE(REPLACE(?attr_name, " ", "_"),":", "_"))) as ?attr_uri)
    }       
"""
object_attr_property_cq = """
    prefix oced: <https://w3id.org/ocedo/core#>
    prefix ext: <https://w3id.org/ocedo/ext#>
    prefix aux: <https://w3id.org/ocedo/aux#>
    prefix res: <https://w3id.org/ocedo/resource/>
    prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    prefix owl: <http://www.w3.org/2002/07/owl#>
    prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>

    CONSTRUCT {
        ?attr_uri a owl:datatypeProperty ;
            rdfs:subPropertyOf oced:object_attribute .

    } WHERE {
        ?object a oced:Object ;
            aux:hasObjectAttribute/aux:object_attribute ?attr_name .
        BIND( IRI(CONCAT("https://w3id.org/ocedo/ext#",REPLACE(REPLACE(?attr_name, " ", "_"),":", "_"))) as ?attr_uri)
    }       
"""

In [ ]:


if use_endpoint:
    temp_data_result += run_construct_query_endpoint(build_query_superclass(event_attr_cq))
    temp_data_result += run_construct_query_endpoint(build_query_superclass(event_type_cq))
    temp_data_result += run_construct_query_endpoint(build_query_superclass(object_attr_cq))
    
    temp_ext_result += run_construct_query_endpoint(build_query_superclass(event_attr_property_cq))
    temp_ext_result += run_construct_query_endpoint(build_query_superclass(event_type_class_cq))
    temp_ext_result += run_construct_query_endpoint(build_query_superclass(object_attr_property_cq))
else:
    temp_data_result += run_construct_query_rdflib(build_query_superclass(event_attr_cq))
    temp_data_result += run_construct_query_rdflib(build_query_superclass(event_type_cq))
    temp_data_result += run_construct_query_rdflib(build_query_superclass(object_attr_cq))
    
    temp_ext_result += run_construct_query_rdflib(build_query_superclass(event_attr_property_cq))
    temp_ext_result += run_construct_query_rdflib(build_query_superclass(event_type_class_cq))
    temp_ext_result += run_construct_query_rdflib(build_query_superclass(object_attr_property_cq))

In [ ]:
# insert query for use-case superclasses
add_superclass_cq = """
    prefix oced: <https://w3id.org/ocedo/core#>
    prefix ext: <https://w3id.org/ocedo/ext#>
    prefix aux: <https://w3id.org/ocedo/aux#>
    prefix res: <https://w3id.org/ocedo/resource/>
    prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    prefix owl: <http://www.w3.org/2002/07/owl#>
    prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>

    INSERT {
        <$event_superclass> a owl:Class ; 
            rdfs:subClassOf oced:Event .
        <$object_superclass> a owl:Class; 
            rdfs:subClassOf oced:Object .
    } WHERE {}
"""
# prepare construct query for object attribute enhancement
aux_cleaner_cq = """
    prefix oced: <https://w3id.org/ocedo/core#>
    prefix ext: <https://w3id.org/ocedo/ext#>
    prefix aux: <https://w3id.org/ocedo/aux#>
    prefix res: <https://w3id.org/ocedo/resource/>
    prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    prefix owl: <http://www.w3.org/2002/07/owl#>
    prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>

    DELETE {
        ?s ?p1 ?o1
    }
    WHERE {
        ?s ?p ?o .
    	?s ?p1 ?o1
        FILTER (
            STRSTARTS(STR(?o), "https://w3id.org/ocedo/aux#")
        )
    }
"""

In [46]:
input_graph += temp_data_result
input_graph += temp_ext_result
ontology_graph +=temp_ext_result

# clean up
input_graph.update(aux_cleaner_cq)
triples_to_remove = [(s, p, o) for s, p, o in input_graph if 
    str(s).startswith(aux) or 
    str(p).startswith(aux) or 
    str(o).startswith(aux) 
    ]

for triple in triples_to_remove:
    input_graph.remove(triple)
input_graph.update(build_query_superclass(add_superclass_cq))

input_graph.serialize(destination=OUTPUT_FILENAME, format='ttl')

<Graph identifier=Nc26bca86e94f46f99dfe90ec857c18ec (<class 'rdflib.graph.Graph'>)>

In [ ]:
ontology_graph +=temp_ext_result
# clean up
ontology_graph.update(aux_cleaner_cq)
triples_to_remove = [(s, p, o) for s, p, o in ontology_graph if 
    str(s).startswith(aux) or 
    str(p).startswith(aux) or 
    str(o).startswith(aux) 
    ]
for triple in triples_to_remove:
    ontology_graph.remove(triple)
ontology_graph.serialize(destination=ext_FILENAME, format='ttl')

<Graph identifier=Nfec9910cfb3149a1a9e40262f604c937 (<class 'rdflib.graph.Graph'>)>